In [57]:
import pandas as pd 
import sys
sys.path.append("./starter-kit")

from validation_structure import normaliser, valider 
df = pd.read_csv("tva_nettoye.csv")
df_resultats = pd.read_csv("resultats_vies.csv")
df_initial = pd.read_csv(
    "./starter-kit/data/numeros_tva.csv",
    dtype=str,
    keep_default_na=False 
)

In [58]:
#verifier le techechement des fichiers 
print("Nombre de lignes nettoyées :", len(df))
print("Nombre de résultats VIES :", len(df_resultats))

Nombre de lignes nettoyées : 5802
Nombre de résultats VIES : 2


In [59]:
# Compter les numéros valides
nb_valides = (df_resultats["isValid"] == True).sum()

# Compter les numéros invalides
nb_invalides = (df_resultats["isValid"] == False).sum()

# Compter les résultats sans verdict (cellule vide, NaN)
nb_non_tranches = df_resultats["isValid"].isna().sum()

print("Numéros valides :", nb_valides)
print("Numéros invalides :", nb_invalides)
print("Numéros non tranchés :", nb_non_tranches)

Numéros valides : 0
Numéros invalides : 2
Numéros non tranchés : 0


In [60]:
#rassembler les résultats dans un tableau
rapport = pd.DataFrame({
    "Catégorie": ["Valides", "Invalides", "Non tranchés"],
    "Nombre": [nb_valides, nb_invalides, nb_non_tranches]
})
rapport 

,Catégorie,Nombre
0,Valides,0
1,Invalides,2
2,Non tranchés,0


In [61]:
# Charger le fichier original
df_initial = pd.read_csv(
    "./starter-kit/data/numeros_tva.csv",
    dtype=str,
    keep_default_na=False
)

# Normaliser les numéros
df_initial["numero_normalise_tva"] = df_initial["numero_tva"].apply(normaliser)

# Appliquer la validation
df_initial["validation"] = df_initial["numero_normalise_tva"].apply(valider)
df_initial


,id,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie,numero_normalise_tva,validation
0,1,Nord Technologies GmbH,DK,DK73224645,2025-07-27,portail_client,DK73224645,"(False, cle_invalide)"
1,2,Solaris Technologies SARL,PT,PT787327239,2024-03-29,reprise_erp,PT787327239,"(True, ok)"
2,3,Verso Consulting NV,LU,LU71043234,2024-02-16,crm,LU71043234,"(True, ok)"
3,4,Lumen Distribution Sp. z o.o.,SE,SE.751298146001,2024-02-27,import_fournisseur,SE751298146001,"(True, ok)"
4,5,Textil Services Oy,NL,NL505862176B88,2025-04-14,crm,NL505862176B88,"(False, cle_invalide)"
...,...,...,...,...,...,...,...,...
9995,9996,Cobalt Trading A/S,PT,PT714332224,2025-01-19,import_fournisseur,PT714332224,"(True, ok)"
9996,9997,Delta Trading Oy,FR,FR60801776699,2025-04-18,crm,FR60801776699,"(True, ok)"
9997,9998,Corvus Industries SARL,SE,SE96633532830,2024-09-12,reprise_erp,SE96633532830,"(False, cle_invalide)"
9998,9999,Prima Distribution AB,LU,LU.98154452,2024-02-15,reprise_erp,LU98154452,"(True, ok)"


In [62]:
# Compter les numéros par motif de rejet
motifs_rejet = df_initial[
    df_initial["validation"].str[0] == False
    # Récupère le deuxième élément du tuple (resultat[1]), c'est-à-dire le motif de validation avec lamda 
]["validation"].apply(lambda resultat: resultat[1]).value_counts()

motifs_rejet

validation
cle_invalide        2605
pays_non_couvert    1090
vide                 174
trop_court            48
Name: count, dtype: int64

In [63]:
# Le motif de rejet le plus fréquent est "cle_invalide", avec 2 605 numéros.
# 1 090 numéros concernent des pays non couverts par notre module.

In [64]:
nb_doublons = df_initial["numero_normalise_tva"].duplicated().sum()
print("Nombre de doublons :", nb_doublons)

Nombre de doublons : 696


In [65]:
# Le fichier initial contient 696 doublons de numéros de TVA.

In [66]:
# rapport complet avec tous les catégorie 
rapport_complet = pd.DataFrame({
    "Catégorie": [
        "Valides (VIES)",
        "Invalides (VIES)",
        "Non tranchés (VIES)",
        "Doublons",
        "Clé invalide",
        "Pays non couvert",
        "Numéro vide",
        "Numéro trop court"
    ],
    "Nombre": [
        nb_valides,
        nb_invalides,
        nb_non_tranches,
        nb_doublons,
        motifs_rejet.get("cle_invalide", 0),
        motifs_rejet.get("pays_non_couvert", 0),
        motifs_rejet.get("vide", 0),
        motifs_rejet.get("trop_court", 0)
    ]
})

rapport_complet

,Catégorie,Nombre
0,Valides (VIES),0
1,Invalides (VIES),2
2,Non tranchés (VIES),0
3,Doublons,696
4,Clé invalide,2605
5,Pays non couvert,1090
6,Numéro vide,174
7,Numéro trop court,48


In [67]:
# Ajouter une colonne indiquant si le numéro est rejeté dans le df initial 
df_initial["est_rejete"] = df_initial["validation"].apply(
    lambda resultat: resultat[0] == False
)
df_initial

,id,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie,numero_normalise_tva,validation,est_rejete
0,1,Nord Technologies GmbH,DK,DK73224645,2025-07-27,portail_client,DK73224645,"(False, cle_invalide)",True
1,2,Solaris Technologies SARL,PT,PT787327239,2024-03-29,reprise_erp,PT787327239,"(True, ok)",False
2,3,Verso Consulting NV,LU,LU71043234,2024-02-16,crm,LU71043234,"(True, ok)",False
3,4,Lumen Distribution Sp. z o.o.,SE,SE.751298146001,2024-02-27,import_fournisseur,SE751298146001,"(True, ok)",False
4,5,Textil Services Oy,NL,NL505862176B88,2025-04-14,crm,NL505862176B88,"(False, cle_invalide)",True
...,...,...,...,...,...,...,...,...,...
9995,9996,Cobalt Trading A/S,PT,PT714332224,2025-01-19,import_fournisseur,PT714332224,"(True, ok)",False
9996,9997,Delta Trading Oy,FR,FR60801776699,2025-04-18,crm,FR60801776699,"(True, ok)",False
9997,9998,Corvus Industries SARL,SE,SE96633532830,2024-09-12,reprise_erp,SE96633532830,"(False, cle_invalide)",True
9998,9999,Prima Distribution AB,LU,LU.98154452,2024-02-15,reprise_erp,LU98154452,"(True, ok)",False


In [68]:

# Calculer le nombre de saisies et de rejets par canal avec agg appliquer plusieurs calculs sur les données regroupées 

analyse_canaux = df_initial.groupby("source_saisie").agg(
    nombre_total=("numero_tva", "count"),
    nombre_rejets=("est_rejete", "sum") 
)
# Calculer le taux de rejet en pourcentage
analyse_canaux["taux_rejet_pct"] = (
    analyse_canaux["nombre_rejets"] / analyse_canaux["nombre_total"] * 100
).round(2)

analyse_canaux

,nombre_total,nombre_rejets,taux_rejet_pct
source_saisie,,,
crm,2019,769,38.09
import_fournisseur,1944,776,39.92
portail_client,1990,770,38.69
reprise_erp,2071,822,39.69
saisie_manuelle,1976,780,39.47


In [69]:

# Comparer le nombre de numéros et de rejets par pays
analyse_pays = df_initial.groupby("pays_declare").agg(
    nombres_total=("numero_tva", "count"),
    numbre_rejets =("est_rejete", "sum")
    )
analyse_pays

,nombres_total,numbre_rejets
pays_declare,,
BE,959,337
DK,961,344
FI,902,343
FR,974,342
GB,104,104
IT,947,352
LU,956,326
NL,947,359
PL,939,311


In [70]:
#date_saisie
# Extraire le mois et l'année de la date de saisie
df_initial["periode"] = pd.to_datetime(df_initial["date_saisie"]).dt.to_period("M")

analyse_pariode = df_initial.groupby("date_saisie").agg(
    nombres_total=("numero_tva", "count"),
    numbre_rejets =("est_rejete", "sum")
    )
analyse_pariode

,nombres_total,numbre_rejets
date_saisie,,
2024-01-01,17,2
2024-01-02,23,8
2024-01-03,16,3
2024-01-04,18,8
2024-01-05,17,5
...,...,...
2025-09-27,24,14
2025-09-28,12,2
2025-09-29,8,4
